# 01 · Clase ternaria y calidad de datos (2019)

**Etapa 1 del plan.** Este notebook:

- parte del dataset original de la cátedra (`competencia_02_crudo.csv`) y se queda con **2019**, como pide la consigna;
- agrega la clase `clase_ternaria`, con el mismo criterio que el código oficial de la cátedra (`z470`);
- verifica todo contra valores conocidos: si algo no coincide, se frena;
- guarda el dataset base en parquet;
- diagnostica lo que afecta a la historia: **meses rotos** y **cobertura** (cuántos clientes tienen h meses de historia).

**No** limpia ni corrige variables, **no** arma lags ni tendencias y **no** entrena. Los meses rotos solo se detectan y se registran: cómo tratarlos se decide en la etapa siguiente.

## 1. Parámetros

Las rutas son relativas a la carpeta del notebook (`feature_historico/notebooks/`).

- `CRUDO` es **lo único que cada integrante tiene que ajustar** si guarda el dataset en otro lugar (por ejemplo, en Google Cloud: `~/buckets/b1/datasets/competencia_02_crudo.csv`).
- `data/` (el parquet) no se versiona; `resultados/` (tablas chicas) sí.

In [1]:
from pathlib import Path
from importlib import metadata
import platform
import time
import warnings

import duckdb
import pandas as pd

# --- Parámetros -------------------------------------------------------------
CRUDO = Path('../../../dataset/competencia_02_crudo.csv')
DATA = Path('../data')
RESULTADOS = Path('../resultados')
REQUIREMENTS = Path('../requirements.txt')

MES_DESDE, MES_HASTA = 201901, 201912   # consigna: solo 2019

SEMILLA_PRIMIGENIA = 979201   # la de la competencia 1 (de ella salen todas las demás semillas)
# -----------------------------------------------------------------------------

BASE_PARQUET = DATA / 'base_2019.parquet'
CLASES_CSV = RESULTADOS / '01_clases_por_mes.csv'
CALIDAD_CSV = RESULTADOS / '01_calidad_por_mes.csv'
ROTOS_CSV = RESULTADOS / '01_meses_rotos.csv'
COBERTURA_CSV = RESULTADOS / '01_cobertura_historia.csv'

# Falso positivo conocido de pandas 2.2 con Python 3.14 (no afecta resultados)
warnings.filterwarnings('ignore', message='.*ChainedAssignmentError.*')

DATA.mkdir(exist_ok=True)
RESULTADOS.mkdir(exist_ok=True)


def es_primo(n):
    return n > 1 and all(n % d for d in range(2, int(n ** 0.5) + 1))


assert CRUDO.exists(), f'No encuentro el crudo en {CRUDO.resolve()}'
assert isinstance(SEMILLA_PRIMIGENIA, int) and 100_000 <= SEMILLA_PRIMIGENIA <= 999_999 \
    and es_primo(SEMILLA_PRIMIGENIA), 'SEMILLA_PRIMIGENIA tiene que ser un primo de 6 dígitos'

## 2. Valores esperados

Se verificaron sobre el crudo el 2026-11-02 (`fichas/etapa_1_clase_calidad.md`). El notebook se **frena** si algo no coincide, para no seguir con datos distintos de los esperados.

Las clases siguen el criterio de la cátedra: un cliente que falta uno o más meses y después vuelve cuenta como `BAJA+1` en el mes anterior al hueco.

In [2]:
ESPERADO_ARCHIVO = {'filas': 4_707_495, 'columnas': 154, 'clientes': 185_094}
ESPERADO_2019 = {'filas': 1_563_565, 'clientes': 146_623, 'duplicados': 0}

ESPERADO_FILAS_POR_MES = {
    201901: 123_916, 201902: 125_020, 201903: 125_622, 201904: 126_116,
    201905: 126_763, 201906: 128_370, 201907: 129_853, 201908: 131_775,
    201909: 133_419, 201910: 135_633, 201911: 137_523, 201912: 139_555,
}

# None = sin clase (falta el futuro para saberla)
ESPERADO_CLASES = {
    201901: {'BAJA+1': 647, 'BAJA+2': 697, 'CONTINUA': 122_572},
    201902: {'BAJA+1': 700, 'BAJA+2': 696, 'CONTINUA': 123_624},
    201903: {'BAJA+1': 700, 'BAJA+2': 774, 'CONTINUA': 124_148},
    201904: {'BAJA+1': 776, 'BAJA+2': 575, 'CONTINUA': 124_765},
    201905: {'BAJA+1': 576, 'BAJA+2': 658, 'CONTINUA': 125_529},
    201906: {'BAJA+1': 665, 'BAJA+2': 589, 'CONTINUA': 127_116},
    201907: {'BAJA+1': 595, 'BAJA+2': 656, 'CONTINUA': 128_602},
    201908: {'BAJA+1': 659, 'BAJA+2': 543, 'CONTINUA': 130_573},
    201909: {'BAJA+1': 548, 'BAJA+2': 615, 'CONTINUA': 132_256},
    201910: {'BAJA+1': 621, 'BAJA+2': 632, 'CONTINUA': 134_380},
    201911: {'BAJA+1': 634, None: 136_889},
    201912: {None: 139_555},
}

# IDs que Python o R pueden escribir en notación científica; veces que aparecen en 2019
ESPERADO_IDS = {48_000_000: 12, 31_100_000: 4}

# Cobertura (% de clientes del mes que ya estaban h meses antes), para h = 2..6
ESPERADO_COBERTURA_201910 = {2: 96.3, 3: 94.4, 4: 92.9, 5: 91.3, 6: 90.2}

## 3. Carga del crudo

- **Por qué se declara el tipo del ID:** el crudo trae `4.8e+07` en 32 filas (es el cliente 48000000). Sin declararlo, DuckDB lee `numero_de_cliente` como decimal; con `types={'numero_de_cliente': 'BIGINT'}` queda entero.
- **`sample_size=-1`:** DuckDB recorre el archivo entero antes de decidir el tipo de cada columna.
- Primero se controla el **archivo completo** (4,7 M de filas, de 201901 a 202108) y después se carga **solo 2019**, para no tener en memoria lo que no se usa.

In [3]:
t0 = time.time()
con = duckdb.connect()
lector = f"read_csv('{CRUDO.as_posix()}', sample_size=-1, types={{'numero_de_cliente': 'BIGINT'}})"

filas, clientes = con.execute(f"select count(*), count(distinct numero_de_cliente) from {lector}").fetchone()
columnas = len(con.execute(f"describe select * from {lector}").fetchall())
print(f'Archivo: {filas:,} filas · {columnas} columnas · {clientes:,} clientes · {time.time() - t0:.0f} s')
assert {'filas': filas, 'columnas': columnas, 'clientes': clientes} == ESPERADO_ARCHIVO

Archivo: 4,707,495 filas · 154 columnas · 185,094 clientes · 150 s


In [4]:
t0 = time.time()
con.execute(f"""
    create or replace table crudo as
    select * from {lector}
    where foto_mes between {MES_DESDE} and {MES_HASTA}
""")

filas, clientes = con.execute("select count(*), count(distinct numero_de_cliente) from crudo").fetchone()
duplicados = con.execute("""
    select count(*) from (
        select numero_de_cliente, foto_mes from crudo group by all having count(*) > 1)
""").fetchone()[0]
por_mes = dict(con.execute("select foto_mes, count(*) from crudo group by 1 order by 1").fetchall())

print(f'2019: {filas:,} filas · {clientes:,} clientes · {duplicados} claves duplicadas · {time.time() - t0:.0f} s')
assert {'filas': filas, 'clientes': clientes, 'duplicados': duplicados} == ESPERADO_2019
assert por_mes == ESPERADO_FILAS_POR_MES

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2019: 1,563,565 filas · 146,623 clientes · 0 claves duplicadas · 78 s


## 4. Clase ternaria

Es la misma lógica que el código oficial de la cátedra (`z470`), escrita en SQL, y la misma del `01` de la competencia 1. Se calcula **sobre 2019 ya filtrado**: así 201911 queda solo con `BAJA+1` y 201912 sin clase, como dice la consigna.

1. **Período consecutivo:** `p0 = año × 12 + mes`. Así el mes siguiente es siempre `p0 + 1`, incluso de diciembre a enero.
2. **Para cada cliente** se miran sus dos períodos siguientes en el dataset: `p1` y `p2`, con `lead`.
3. **Se asigna la clase:**

| Clase | Condición |
|---|---|
| `BAJA+2` | está el mes siguiente (`p1 = p0 + 1`) pero no el subsiguiente (`p2` no existe o es mayor que `p0 + 2`) |
| `BAJA+1` | no está el mes siguiente (`p1` no existe o es mayor que `p0 + 1`) |
| `CONTINUA` | el resto, siempre que haya dos meses de futuro para saberlo |
| `NULL` | falta el futuro: 201912 entero, y 201911 salvo los `BAJA+1` |

**La lógica de tres valores de SQL:** si `p2` es `NULL`, `p2 > p0 + 2` da `NULL` y no `TRUE`. Por eso se pregunta explícitamente `p2 is null`. Si no, la fila caería por defecto en otra clase.

In [5]:
ultimo = con.execute("select max((foto_mes // 100) * 12 + foto_mes % 100) from crudo").fetchone()[0]

con.execute(f"""
    create or replace table con_clase as
    with periodos as (
        select *, (foto_mes // 100) * 12 + foto_mes % 100 as p0
        from crudo
    ),
    futuros as (
        select *,
               lead(p0, 1) over w as p1,
               lead(p0, 2) over w as p2
        from periodos
        window w as (partition by numero_de_cliente order by p0)
    )
    select * exclude (p0, p1, p2),
           case
               when p0 < {ultimo} - 1 and p1 = p0 + 1 and (p2 is null or p2 > p0 + 2) then 'BAJA+2'
               when p0 < {ultimo}     and (p1 is null or p1 > p0 + 1)                  then 'BAJA+1'
               when p0 < {ultimo} - 1                                                    then 'CONTINUA'
               else null
           end as clase_ternaria
    from futuros
""")
con.execute("drop table crudo")

tabla_clases = con.execute("""
    select foto_mes, clase_ternaria, count(*) as n
    from con_clase group by all order by all
""").df()
pivote_clases = (tabla_clases.assign(clase_ternaria=tabla_clases['clase_ternaria'].fillna('sin clase'))
                             .pivot(index='foto_mes', columns='clase_ternaria', values='n')
                             .fillna(0).astype(int))
pivote_clases.to_csv(CLASES_CSV)
pivote_clases

clase_ternaria,BAJA+1,BAJA+2,CONTINUA,sin clase
foto_mes,,,,
201901,647,697,122572,0
201902,700,696,123624,0
201903,700,774,124148,0
201904,776,575,124765,0
201905,576,658,125529,0
201906,665,589,127116,0
201907,595,656,128602,0
201908,659,543,130573,0
201909,548,615,132256,0


In [6]:
def clases_por_mes(tabla):
    """{foto_mes: {clase: n}}, con None para las filas sin clase."""
    salida = {}
    for mes, clase, n in tabla[['foto_mes', 'clase_ternaria', 'n']].itertuples(index=False):
        salida.setdefault(int(mes), {})[None if pd.isna(clase) else clase] = int(n)
    return salida


assert clases_por_mes(tabla_clases) == ESPERADO_CLASES, 'La tabla de clases no coincide con la esperada'
tasa_b2 = (pivote_clases['BAJA+2'] / pivote_clases.sum(axis=1)).loc[:201910].mul(100).round(2)
print('Clases OK: coinciden con el criterio de la cátedra')
print('% de BAJA+2 por mes:', tasa_b2.to_dict())

Clases OK: coinciden con el criterio de la cátedra
% de BAJA+2 por mes: {201901: 0.56, 201902: 0.56, 201903: 0.62, 201904: 0.46, 201905: 0.52, 201906: 0.46, 201907: 0.51, 201908: 0.41, 201909: 0.46, 201910: 0.47}


## 5. Tipos

- `numero_de_cliente` queda en `int64` (`BIGINT`);
- `foto_mes` pasa a entero de 32 bits;
- `clase_ternaria` queda como texto;
- **todo el resto pasa a `float32`** (`FLOAT` en DuckDB).

**Por qué:** reduce a la mitad la memoria de las 152 variables, y después vienen las variables de historia. LightGBM trabaja internamente en `float32`, así que el modelo no pierde nada. Los `NULL` se mantienen: no hay ningún `fillna` ni ninguna imputación.

In [7]:
columnas_crudo = [c[0] for c in con.execute("describe con_clase").fetchall()]
claves = ['numero_de_cliente', 'foto_mes', 'clase_ternaria']
variables = [c for c in columnas_crudo if c not in claves]

select_tipado = ',\n    '.join(
    ['numero_de_cliente::BIGINT as numero_de_cliente', 'foto_mes::INTEGER as foto_mes']
    + [f'"{c}"::FLOAT as "{c}"' for c in variables]
    + ['clase_ternaria']
)
con.execute(f"create or replace table base as select\n    {select_tipado}\nfrom con_clase order by foto_mes, numero_de_cliente")
con.execute("drop table con_clase")

tipos = con.execute("describe base").df().set_index('column_name')['column_type']
print(tipos.value_counts().to_string())
assert tipos['numero_de_cliente'] == 'BIGINT' and tipos['foto_mes'] == 'INTEGER'
assert tipos['clase_ternaria'] == 'VARCHAR'
assert (tipos[variables] == 'FLOAT').all() and len(variables) == 152

column_type
FLOAT      152
BIGINT       1
INTEGER      1
VARCHAR      1


## 6. Controles del dataset base

- 201911 y 201912: solo `BAJA+1` o sin clase;
- IDs: enteros, y los que se escriben en notación científica aparecen las veces esperadas.

In [8]:
for mes, esperado in [(201911, {'BAJA+1': 634, 'NULL': 136_889}), (201912, {'NULL': 139_555})]:
    obtenido = dict(con.execute(f"""select coalesce(clase_ternaria, 'NULL'), count(*) from base
                                     where foto_mes = {mes} group by 1""").fetchall())
    print(mes, obtenido)
    assert obtenido == esperado

for id_, veces in ESPERADO_IDS.items():
    n = con.execute(f"select count(*) from base where numero_de_cliente = {id_}").fetchone()[0]
    print(f'{id_}: {n} filas')
    assert n == veces

no_enteros = con.execute("select count(*) from base where numero_de_cliente <> round(numero_de_cliente)").fetchone()[0]
assert no_enteros == 0
print('Controles OK')

201911 {'NULL': 136889, 'BAJA+1': 634}
201912 {'NULL': 139555}
48000000: 12 filas
31100000: 4 filas
Controles OK


## 7. Persistencia en parquet

Se guarda el dataset base, **ordenado por mes y cliente**, y se relee para confirmar que quedó igual: filas, tipos y la tabla de clases. El orden importa: con los mismos datos en otro orden, LightGBM da otro modelo.

Los notebooks siguientes leen este archivo, no el crudo. `data/` no se versiona.

In [9]:
t0 = time.time()
con.execute(f"copy base to '{BASE_PARQUET.as_posix()}' (format parquet, compression zstd)")
print(f'Guardado: {BASE_PARQUET} · {BASE_PARQUET.stat().st_size / 1e6:.0f} MB · {time.time() - t0:.0f} s')

releido = f"read_parquet('{BASE_PARQUET.as_posix()}')"
filas_p = con.execute(f"select count(*) from {releido}").fetchone()[0]
tipos_p = con.execute(f"describe select * from {releido}").df().set_index('column_name')['column_type']
tabla_p = con.execute(f"""select foto_mes, clase_ternaria, count(*) as n
                          from {releido} group by all order by all""").df()

assert filas_p == ESPERADO_2019['filas']
assert tipos_p.equals(tipos), 'Los tipos del parquet no coinciden con los de la tabla base'
assert clases_por_mes(tabla_p) == ESPERADO_CLASES
print(f'Parquet OK: {filas_p:,} filas · {len(tipos_p)} columnas · clases iguales')

Guardado: ../data/base_2019.parquet · 186 MB · 1 s
Parquet OK: 1,563,565 filas · 155 columnas · clases iguales


## 8. Calidad por mes y meses rotos

Para cada variable y cada mes, el **% de nulos** y el **% de ceros**, los dos sobre todas las filas del mes: las "dos reglas de oro" de la Clase 08.
- **Un salto de nulos** suele ser un mal *left join*.
- **Un salto de ceros** suele ser una imputación por defecto.

Se guarda completa en `resultados/01_calidad_por_mes.csv`. Abajo se listan las variables donde algún mes se aparta más de 5 puntos de la mediana de 2019.

In [10]:
t0 = time.time()
agregados = ',\n    '.join(
    f'avg(("{c}" is null)::INT) as "nulos__{c}", avg(coalesce("{c}" = 0, false)::INT) as "ceros__{c}"'
    for c in variables
)
ancho = con.execute(f"select foto_mes,\n    {agregados}\nfrom base group by 1 order by 1").df()

calidad = (ancho.melt(id_vars='foto_mes', var_name='medida_variable', value_name='proporcion')
                .assign(medida=lambda d: d['medida_variable'].str.split('__').str[0],
                        variable=lambda d: d['medida_variable'].str.split('__').str[1])
                .pivot_table(index=['variable', 'medida'], columns='foto_mes', values='proporcion')
                .mul(100).round(2))
calidad.to_csv(CALIDAD_CSV)
print(f'Guardado: {CALIDAD_CSV} · {len(calidad)} filas · {time.time() - t0:.0f} s')

desvio = calidad.sub(calidad.median(axis=1), axis=0).abs().max(axis=1)
saltos = (calidad[desvio > 5].assign(max_desvio_pp=desvio[desvio > 5])
                             .sort_values('max_desvio_pp', ascending=False))
print(f'{len(saltos)} casos con algún mes a más de 5 puntos de la mediana')
saltos

Guardado: ../resultados/01_calidad_por_mes.csv · 304 filas · 0 s
35 casos con algún mes a más de 5 puntos de la mediana


,foto_mes,201901,201902,201903,201904,201905,201906,201907,201908,201909,201910,201911,201912,max_desvio_pp
variable,medida,,,,,,,,,,,,,
mrentabilidad_annual,ceros,0.22,0.23,0.21,0.19,100.00,0.21,0.22,0.23,0.25,100.00,0.26,0.25,99.770
mrentabilidad,ceros,0.24,0.26,0.24,0.22,100.00,0.25,0.25,0.27,0.28,100.00,0.31,0.30,99.735
mpasivos_margen,ceros,1.20,1.24,1.23,1.21,100.00,1.18,1.10,1.39,1.07,100.00,1.29,1.23,98.770
ccomisiones_otras,ceros,2.35,2.69,2.68,2.66,100.00,2.89,2.68,2.84,3.42,100.00,2.90,2.80,97.180
mcomisiones_otras,ceros,2.35,2.70,2.68,2.66,100.00,2.90,2.68,2.84,3.43,100.00,2.91,2.80,97.180
mcomisiones,ceros,2.36,2.70,2.68,2.66,100.00,2.90,2.68,2.84,3.43,100.00,2.91,2.80,97.180
mactivos_margen,ceros,3.54,3.87,3.97,4.09,100.00,3.86,4.41,4.41,4.25,100.00,4.48,4.44,95.670
ctarjeta_visa_debitos_automaticos,ceros,33.00,33.17,33.33,100.00,33.22,33.38,33.04,33.40,33.68,33.68,34.08,34.56,66.610
mttarjeta_visa_debitos_automaticos,ceros,33.00,33.17,33.33,100.00,33.22,33.38,33.04,33.40,33.68,33.68,34.08,34.56,66.610


**Meses rotos** (decisión 4 de la ficha). Un mes cuenta como roto en una variable si:
- tiene **≥ 95 % de ceros o de nulos**, y
- **los meses vecinos** (el anterior y el siguiente que existan) están **por debajo del 50 %**.

Es un mes aislado. Por diseño, la regla deja afuera las variables que **empiezan a medirse** a mitad de año (transferencias recibidas, app móvil) y los cambios de régimen.

Además se agregan **a mano** dos casos que la regla no captura:
- `Visa_fultimo_cierre` y `Master_fultimo_cierre` en 201907 (63–66 % en cero, contra 0 % el resto);
- las 4 variables de descuentos en 201910, el mes de test (pasan de 85–96 % a 100 %).

La lista se guarda en `resultados/01_meses_rotos.csv`. Se usa en la etapa siguiente, donde se decide cómo tratarlos.

In [11]:
UMBRAL_ROTO, UMBRAL_VECINOS = 95, 50

meses = list(calidad.columns)
regla = []
for (variable, medida), fila in calidad.iterrows():
    for i, mes in enumerate(meses):
        vecinos = [fila[meses[j]] for j in (i - 1, i + 1) if 0 <= j < len(meses)]
        if fila[mes] >= UMBRAL_ROTO and all(v < UMBRAL_VECINOS for v in vecinos):
            regla.append((variable, medida, mes, fila[mes], 'regla'))

A_MANO = [
    ('Visa_fultimo_cierre', 'ceros', 201907), ('Master_fultimo_cierre', 'ceros', 201907),
    ('mcajeros_propios_descuentos', 'ceros', 201910), ('ccajeros_propios_descuentos', 'ceros', 201910),
    ('ctarjeta_visa_descuentos', 'ceros', 201910), ('mtarjeta_visa_descuentos', 'ceros', 201910),
]
a_mano = [(v, m, mes, calidad.loc[(v, m), mes], 'a mano') for v, m, mes in A_MANO]

rotos = pd.DataFrame(regla + a_mano, columns=['variable', 'medida', 'foto_mes', 'porcentaje', 'origen'])
rotos.to_csv(ROTOS_CSV, index=False)

ESPERADO_REGLA = {
    (v, 'ceros', mes)
    for mes in (201905, 201910)
    for v in ('mrentabilidad', 'mrentabilidad_annual', 'mpasivos_margen', 'mactivos_margen',
              'mcomisiones', 'mcomisiones_otras', 'ccomisiones_otras')
} | {('chomebanking_transacciones', 'ceros', 201910),
     ('ctarjeta_visa_debitos_automaticos', 'ceros', 201904),
     ('mttarjeta_visa_debitos_automaticos', 'ceros', 201904)}

assert {(v, m, mes) for v, m, mes, *_ in regla} == ESPERADO_REGLA, 'La lista de meses rotos no es la esperada'
assert len(rotos) == 23
print(f'Meses rotos: {len(regla)} por la regla + {len(a_mano)} a mano = {len(rotos)}')
rotos.pivot_table(index='foto_mes', values='variable', aggfunc='count').rename(columns={'variable': 'variables rotas'})

Meses rotos: 17 por la regla + 6 a mano = 23


,variables rotas
foto_mes,
201904,2
201905,7
201907,2
201910,12


## 9. Cobertura de historia

Para cada mes y cada h de 2 a 6: el **% de clientes del mes que ya estaban en el banco h meses antes**. Muestra cuánta historia existe de verdad en cada mes. Los ceros vienen del borde del dataset (empieza en 201901), no de los clientes.

In [12]:
cobertura = con.execute("""
    with p as (select numero_de_cliente as id, foto_mes, (foto_mes // 100) * 12 + foto_mes % 100 as idx from base)
    select a.foto_mes,
           round(100 * avg((b2.id is not null)::INT), 1) as h2,
           round(100 * avg((b3.id is not null)::INT), 1) as h3,
           round(100 * avg((b4.id is not null)::INT), 1) as h4,
           round(100 * avg((b5.id is not null)::INT), 1) as h5,
           round(100 * avg((b6.id is not null)::INT), 1) as h6
    from p a
    left join p b2 on b2.id = a.id and b2.idx = a.idx - 2
    left join p b3 on b3.id = a.id and b3.idx = a.idx - 3
    left join p b4 on b4.id = a.id and b4.idx = a.idx - 4
    left join p b5 on b5.id = a.id and b5.idx = a.idx - 5
    left join p b6 on b6.id = a.id and b6.idx = a.idx - 6
    group by a.foto_mes order by a.foto_mes
""").df().set_index('foto_mes')
cobertura.to_csv(COBERTURA_CSV)

fila_201910 = {h: cobertura.loc[201910, f'h{h}'] for h in range(2, 7)}
assert fila_201910 == ESPERADO_COBERTURA_201910, fila_201910
cobertura

,h2,h3,h4,h5,h6
foto_mes,,,,,
201901,0.0,0.0,0.0,0.0,0.0
201902,0.0,0.0,0.0,0.0,0.0
201903,97.6,0.0,0.0,0.0,0.0
201904,98.0,96.6,0.0,0.0,0.0
201905,97.9,96.9,95.5,0.0,0.0
201906,97.2,96.3,95.3,93.9,0.0
201907,96.7,95.6,94.7,93.7,92.3
201908,96.5,94.8,93.7,92.9,91.9
201909,96.4,94.8,93.2,92.1,91.2


## 10. Entorno

Se escribe `requirements.txt` con las versiones de este entorno, para que el resto del grupo arme el mismo. `pyarrow` no hace falta: los parquet se leen con DuckDB.

In [13]:
PAQUETES = ['duckdb', 'lightgbm', 'optuna', 'pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib',
            'jupyter', 'ipykernel']
lineas = [f'# Python {platform.python_version()}'] + [f'{p}=={metadata.version(p)}' for p in PAQUETES]
REQUIREMENTS.write_text('\n'.join(lineas) + '\n')
print(REQUIREMENTS.read_text())

# Python 3.14.7
duckdb==1.5.5
lightgbm==4.7.0
optuna==4.9.0
pandas==2.2.2
numpy==2.4.3
scipy==1.17.1
scikit-learn==1.8.0
matplotlib==3.10.8
jupyter==1.1.1
ipykernel==7.2.0



## 11. Cierre

- **Verificación de la ficha:** los controles 1 a 9 se cumplen si el notebook llegó hasta acá sin frenarse. El 10 (reproducibilidad) se cumple corriéndolo entero en un kernel nuevo: *Restart* → *Run All*.
- **Registro:** anotar en `decisiones.md` el resultado y cualquier hallazgo nuevo.
- **Lo que sigue:** la etapa 2 (experimento mínimo), con su ficha. El próximo notebook lee `data/base_2019.parquet` y `resultados/01_meses_rotos.csv`.

In [14]:
con.close()